# RAG Retrieval and Generation Evaluation



In [1]:
from openai import OpenAI
from src.rag.answer import build_context, answer_with_rag
import os
from src.rag.retriever import RagRetriever


/home/yihaochen/stitching-project-1haochen/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Base LLM result

In [2]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
question = "What are F1 team shutdown periods?"
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a precise regulations assistant."},
        {"role": "user", "content": question},
    ],
    temperature=0.1,
)
print("GPT-4o-mini answer (no RAG):")
print(response.choices[0].message.content.strip())

GPT-4o-mini answer (no RAG):
In Formula 1, team shutdown periods refer to designated times during the year when teams are required to halt all operations related to car development and racing activities. These shutdowns are part of the regulations set by the FIA (Fédération Internationale de l'Automobile) to ensure that team personnel have a break and to promote a more sustainable work-life balance within the sport.

Typically, the shutdown period occurs during the summer months, often in August, and lasts for a minimum of two weeks. During this time, teams are not allowed to conduct any work on their cars, including design, manufacturing, or testing activities. This regulation applies to all team members, including engineers, mechanics, and other staff.

The specific dates and duration of the shutdown can vary from year to year, and teams must adhere to the guidelines set forth by the FIA. The shutdown periods are intended to help teams manage their resources effectively and to ensure

### RAG Retrieval Results

This session demonstrates **RAG** over chunked regulations:
1) Retrieve relevant chunks from Pinecone (`src.rag.retriever`)  
2) Generate an answer with OpenAI `gpt-4o-mini` (`src.rag.answer`)

Components:
- Retrieval module: `src.rag.retriever.RagRetriever`
- Retrieval CLI: `python -m src.rag.run_retrieval`
- RAG QA CLI: `python -m src.rag.answer`

Important: `BAAI/bge-large-en-v1.5` creates `1024`-dim vectors, so your Pinecone index must also be `1024`-dim.


In [3]:
# Unified config source: .env
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "regs-2026-bge1024")
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "regulations-2026")
PINECONE_HOST = os.getenv("PINECONE_HOST")  # optional; set only if host belongs to this same index

print(f"INDEX_NAME={INDEX_NAME}")
print(f"NAMESPACE={NAMESPACE}")
print(f"PINECONE_HOST={'set' if PINECONE_HOST else 'not set'}")


retriever = RagRetriever(
    index_name=INDEX_NAME,
    host=PINECONE_HOST,
    namespace=NAMESPACE,
    chunks_root="chunks",
)
query = "What are the objectives of the Power Unit Financial Regulations?"
results = retriever.retrieve(query, top_k=3)
print(f"retrieved={len(results)} for query: {query}")

if not results:
    print("No matches returned. Check index/namespace or ingest more chunks.")
else:
    for i, r in enumerate(results, start=1):
        clause = r.metadata.get("parent_clause_id") or r.metadata.get("article_id") or r.metadata.get("appendix_id")
        print(f"[{i}] score={r.score:.4f} clause={clause} source={r.metadata.get('source_file')}")
        print((r.text or "(no text)")[:300])
        print("-" * 80)



INDEX_NAME=regs-2026-bge1024
NAMESPACE=regulations-2026
PINECONE_HOST=not set
retrieved=3 for query: What are the objectives of the Power Unit Financial Regulations?
[1] score=0.7525 clause=E6.2 source=FIA 2026 F1 Regulations - Section E [Financial Regulations - Power Unit Manufacturers] - Iss 03 - 2025-12-10.pdf_by_PaddleOCR-VL_no_strike.chunks.jsonl
E6.2 Clariﬁcation of the Power Unit Financial Regulations

E6.2.1 The CFO of a Power Unit Manufacturer may submit a written request to the Cost Cap Administration in order to clarify the operation or interpretation of these Power Unit Financial Regulations. The Cost Cap Administration will respond i
--------------------------------------------------------------------------------
[2] score=0.7418 clause=E1.2 source=FIA 2026 F1 Regulations - Section E [Financial Regulations - Power Unit Manufacturers] - Iss 03 - 2025-12-10.pdf_by_PaddleOCR-VL_no_strike.chunks.jsonl
E1.2 Objectives

E1.2.1 These Power Unit Financial Regulations define a Powe

In [4]:
# Actual RAG answer generation with OpenAI gpt-4o-mini
# Requires the retrieval cell above (`results`, `query` from `retriever.retrieve`).
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
context = build_context(results, max_context_chars=12000)

if not context.strip():
    print("No retrieval context (empty results or no chunk text). Run the retrieval cell first.")
else:
    answer = answer_with_rag(
        query=query,
        context=context,
        model="gpt-4o-mini",
        client=client,
    )
    print("\nRAG answer:\n")
    print(answer)



RAG answer:

The objectives of the Power Unit Financial Regulations are to:

a. Promote the long-term competitive balance of the Championship in respect of Power Units;

b. Promote the long-term sporting fairness of the Championship in respect of Power Units; and

c. Ensure the long-term financial stability and sustainability of the Power Unit Manufacturers, while preserving the unique technology and engineering challenge of Formula 1 [Chunk 2].


In [5]:
# CLI examples (run in notebook shell)

# Retrieval only (uses .env defaults for index/namespace/host):
# !.venv/bin/python -m src.rag.run_retrieval \
#   --query "What are F1 team shutdown periods?" \
#   --chunks-root chunks \
#   --top-k 3

# Actual RAG answer with OpenAI gpt-4o-mini (also uses .env defaults):
# !.venv/bin/python -m src.rag.answer \
#   --query "What are F1 team shutdown periods?" \
#   --top-k 5 \
#   --llm-model gpt-4o-mini

# You can still override any default explicitly, e.g. --index-name / --namespace / --host

### Advanced Multi-Agent RAG (LangGraph)

Run the new `src.advanced_rag` system (memory agent, planner, retriever, relevance judge, reference resolver, synthesizer, answer checker).

**Answer synthesizer** (final grounded answer only):
- `answer_synthesizer_backend="openai"` — JSON mode via OpenAI (default `gpt-4o-mini`, or set `answer_synthesizer_openai_model`).
- `answer_synthesizer_backend="local_qwen"` + **`qwen_use_base_only=True`** — **base** **Qwen2.5-Instruct** from Hugging Face (no LoRA; first run may download weights to your HF cache).
- `answer_synthesizer_backend="local_qwen"` + adapter path — fine-tuned **Qwen2.5-1.5B** LoRA under `models/` (PEFT; needs `adapter_model*.safetensors`, e.g. after unzipping `models.zip`).

Other agents (memory, planner, judge, etc.) still use OpenAI.

This demo includes:
1) OpenAI synthesizer: two-turn dialogue + citations
2) Base local Qwen synthesizer: same style (GPU recommended; CPU is slow)
3) LoRA Qwen synthesizer (optional, if adapter files exist)
4) Recorded example output for the base-Qwen path (for reports if you skip the heavy cell)

In [6]:
from src.advanced_rag import AdvancedRegulationAssistant

# --- OpenAI answer synthesizer (default): gpt-4o-mini JSON ---
assistant = AdvancedRegulationAssistant(
    llm_model="gpt-4o-mini",
    answer_synthesizer_backend="openai",
    answer_synthesizer_openai_model="gpt-4o-mini",
    top_k=8,
    chunks_root="chunks",
)
print("Synthesizer:", assistant.answer_synthesizer_backend)

q1 = "What are the objectives of the Power Unit Financial Regulations?"
r1 = assistant.ask(q1, session_id="demo-session-openai")

print("Q1:", q1)
print("Resolved Q1:", r1.resolved_query)
print("Query Type:", r1.query_type)
print("Supported:", r1.answer_supported)
print("\nAnswer 1:\n", r1.answer)
print("\nCitations 1:")
for i, c in enumerate(r1.citations, start=1):
    md = c.get("metadata", {})
    clause = md.get("parent_clause_id") or md.get("article_id") or md.get("appendix_id") or ""
    print(f"[Chunk {i}] score={c['score']:.4f} clause={clause} source={md.get('source_file')}")

print("\n" + "=" * 80 + "\n")

q2 = "What about in E1.2.1? Does that apply to all manufacturers?"
r2 = assistant.ask(q2, session_id="demo-session-openai")

print("Q2:", q2)
print("Resolved Q2:", r2.resolved_query)
print("Query Type:", r2.query_type)
print("Supported:", r2.answer_supported)
print("\nAnswer 2:\n", r2.answer)
print("\nCitations 2:")
for i, c in enumerate(r2.citations, start=1):
    md = c.get("metadata", {})
    clause = md.get("parent_clause_id") or md.get("article_id") or md.get("appendix_id") or ""
    print(f"[Chunk {i}] score={c['score']:.4f} clause={clause} source={md.get('source_file')}")


Synthesizer: openai
Q1: What are the objectives of the Power Unit Financial Regulations?
Resolved Q1: What are the objectives of the Power Unit Financial Regulations?
Query Type: definition_lookup
Supported: True

Answer 1:
 The objectives of the Power Unit Financial Regulations are to: a) promote the long-term competitive balance of the Championship in respect of Power Units; b) promote the long-term sporting fairness of the Championship in respect of Power Units; and c) ensure the long-term financial stability and sustainability of the Power Unit Manufacturers, while preserving the unique technology and engineering challenge of Formula 1.

Citations 1:
[Chunk 1] score=0.8298 clause=E1.2 source=FIA 2026 F1 Regulations - Section E [Financial Regulations - Power Unit Manufacturers] - Iss 03 - 2025-12-10.pdf_by_PaddleOCR-VL_no_strike.chunks.jsonl
[Chunk 2] score=1.2000 clause=E1.2 source=FIA 2026 F1 Regulations - Section E [Financial Regulations - Power Unit Manufacturers] - Iss 03 - 202

#### Base local Qwen (no LoRA) as answer synthesizer

`qwen_use_base_only=True` loads **`Qwen/Qwen2.5-1.5B-Instruct`** (or set `qwen_base_model` / env `ADVANCED_RAG_QWEN_BASE_MODEL`). Same two-turn + citation printout as the OpenAI cell above.

If the next cell errors (OOM, no GPU, or offline), use the **example output** cell below for your write-up.

In [7]:
from src.advanced_rag import AdvancedRegulationAssistant

# Base HF model only — no PEFT checkpoint required.
try:
    assistant_bq = AdvancedRegulationAssistant(
        llm_model="gpt-4o-mini",
        answer_synthesizer_backend="local_qwen",
        qwen_use_base_only=True,
        qwen_base_model="Qwen/Qwen2.5-1.5B-Instruct",
        qwen_max_new_tokens=1024,
        top_k=8,
        chunks_root="chunks",
    )
    tag = "local_qwen (base only)" if assistant_bq.qwen_use_base_only else assistant_bq.answer_synthesizer_backend
    print("Synthesizer:", tag)

    q1 = "What are the objectives of the Power Unit Financial Regulations?"
    r1 = assistant_bq.ask(q1, session_id="demo-session-base-qwen")

    print("Q1:", q1)
    print("Resolved Q1:", r1.resolved_query)
    print("Query Type:", r1.query_type)
    print("Supported:", r1.answer_supported)
    print("\nAnswer 1:\n", r1.answer)
    print("\nCitations 1:")
    for i, c in enumerate(r1.citations, start=1):
        md = c.get("metadata", {})
        clause = md.get("parent_clause_id") or md.get("article_id") or md.get("appendix_id") or ""
        print(f"[Chunk {i}] score={c['score']:.4f} clause={clause} source={md.get('source_file')}")

    print("\n" + "=" * 80 + "\n")

    q2 = "What about in E1.2.1? Does that apply to all manufacturers?"
    r2 = assistant_bq.ask(q2, session_id="demo-session-base-qwen")

    print("Q2:", q2)
    print("Resolved Q2:", r2.resolved_query)
    print("Query Type:", r2.query_type)
    print("Supported:", r2.answer_supported)
    print("\nAnswer 2:\n", r2.answer)
    print("\nCitations 2:")
    for i, c in enumerate(r2.citations, start=1):
        md = c.get("metadata", {})
        clause = md.get("parent_clause_id") or md.get("article_id") or md.get("appendix_id") or ""
        print(f"[Chunk {i}] score={c['score']:.4f} clause={clause} source={md.get('source_file')}")
except Exception as exc:
    print("Skip or error running base Qwen (GPU/OOM/network):", type(exc).__name__, exc)

Synthesizer: local_qwen (base only)


`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q1: What are the objectives of the Power Unit Financial Regulations?
Resolved Q1: What are the objectives of the Power Unit Financial Regulations?
Query Type: definition_lookup
Supported: True

Answer 1:
 The objectives of the Power Unit Financial Regulations include promoting the long-term competitive balance of the Championship in respect of Power Units, promoting the long-term sporting fairness of the Championship in respect of Power Units, and ensuring the long-term financial stability and sustainability of the Power Unit Manufacturers, while preserving the unique technology and engineering challenge of Formula 1.

Citations 1:
[Chunk 1] score=0.8298 clause=E1.2 source=FIA 2026 F1 Regulations - Section E [Financial Regulations - Power Unit Manufacturers] - Iss 03 - 2025-12-10.pdf_by_PaddleOCR-VL_no_strike.chunks.jsonl
[Chunk 2] score=1.2000 clause=E1.2 source=FIA 2026 F1 Regulations - Section E [Financial Regulations - Power Unit Manufacturers] - Iss 03 - 2025-12-10.pdf_by_PaddleOC

### Fine-tuned Qwen (LoRA) as answer synthesizer (`local_qwen`)

Same LangGraph pipeline; only **`answer_synthesizer_agent`** uses the LoRA under `models/Qwen2.5-1.5B-lora/...` (base `Qwen/Qwen2.5-1.5B-Instruct`). For **base model only** (no adapter files), use **`qwen_use_base_only=True`** in the section above.

Requires **adapter weight files** in the checkpoint folder (e.g. `adapter_model.safetensors`). If you only have config/tokenizer files, unzip `models.zip` or copy checkpoints from training.

Below: run if weights exist; otherwise skip.

In [8]:
from pathlib import Path

from src.advanced_rag import AdvancedRegulationAssistant

REPO = Path.cwd()
ADAPTER = REPO / "models" / "Qwen2.5-1.5B-lora" 

weight_files = list(ADAPTER.glob("adapter_model*.safetensors")) + list(ADAPTER.glob("*.bin"))
if not weight_files:
    print("Skip: no LoRA weights in", ADAPTER, "(unzip models.zip or train checkpoint).")
else:
    assistant_qwen = AdvancedRegulationAssistant(
        llm_model="gpt-4o-mini",
        answer_synthesizer_backend="local_qwen",
        qwen_adapter_path=str(ADAPTER),
        qwen_max_new_tokens=1024,
        top_k=8,
        chunks_root="chunks",
    )
    print("Synthesizer:", assistant_qwen.answer_synthesizer_backend, "| adapter:", ADAPTER)

    q = "What are the objectives of the Power Unit Financial Regulations?"
    rq = assistant_qwen.ask(q, session_id="demo-qwen-synth")
    print("Q:", q)
    print("Resolved:", rq.resolved_query)
    print("Query type:", rq.query_type)
    print("Supported:", rq.answer_supported)
    print("\nAnswer (Qwen synthesizer):\n", rq.answer)


Synthesizer: local_qwen | adapter: /home/yihaochen/stitching-project-1haochen/models/Qwen2.5-1.5B-lora
Q: What are the objectives of the Power Unit Financial Regulations?
Resolved: What are the objectives of the Power Unit Financial Regulations?
Query type: definition_lookup
Supported: True

Answer (Qwen synthesizer):
 The objectives of the Power Unit Financial Regulations include promoting the long-term competitive balance of the Championship in respect of Power Units, promoting the long-term sporting fairness of the Championship in respect of Power Units, ensuring the long-term financial stability and sustainability of the Power Unit Manufacturers, while preserving the unique technology and engineering challenge of Formula 1.


## Horizontal RAGs evaluations

Same **8 questions** as the compact set in `test_questions.md` (§ “Very good evaluation set”, items 1–8).

Pipelines:

- **a.** Base LLM (no RAG) — `gpt-4o-mini` only.
- **b.** Basic RAG — `RagRetriever` + `answer_with_rag` (`src.rag`).
- **c.** Advanced agentic RAG — **base** local Qwen synthesizer (`qwen_use_base_only=True`; other agents OpenAI).
- **d.** Advanced agentic RAG — **fine-tuned** Qwen LoRA synthesizer (skipped if no adapter weights under `models/`).

**2.** Qualitatively compare outputs (a)→(d): grounding in regulations, specificity, and where the multi-agent + fine-tuned path helps or hurts.

Run the **next cell** to generate side-by-side answers (long run: many LLM calls + optional local model loads).

In [11]:
# Compact evaluation set: test_questions.md — "Very good evaluation set" items 1–8
EVAL_QUESTIONS = [
    "What are the objectives of the Power Unit Financial Regulations?",
    "What is the Power Unit Cost Cap in a manufacturer’s inaugural season?",
    "What is a Reporting Group?",
    "Are employee bonus costs fully excluded, or is there a cap?",
    "Are finance costs excluded from Relevant Costs?",
    "How are Related Party Transactions treated in Relevant Costs?",
    "How are inventories treated when calculating Relevant Costs?",
    "What happens if research and development costs are deferred to a later reporting period?",
]

import os
from pathlib import Path

from openai import OpenAI

from src.rag.answer import build_context, answer_with_rag
from src.rag.retriever import RagRetriever
from src.advanced_rag import AdvancedRegulationAssistant

REPO = Path.cwd()
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "regs-2026-bge1024")
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "regulations-2026")
PINECONE_HOST = os.getenv("PINECONE_HOST")
TOP_K = 8

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
retriever = RagRetriever(
    index_name=INDEX_NAME,
    host=PINECONE_HOST,
    namespace=NAMESPACE,
    chunks_root="chunks",
)

assistant_base_qwen = AdvancedRegulationAssistant(
    llm_model="gpt-4o-mini",
    answer_synthesizer_backend="local_qwen",
    qwen_use_base_only=True,
    qwen_base_model="Qwen/Qwen2.5-1.5B-Instruct",
    qwen_max_new_tokens=1024,
    top_k=TOP_K,
    chunks_root="chunks",
    index_name=INDEX_NAME,
    host=PINECONE_HOST,
    namespace=NAMESPACE,
)

ADAPTER = REPO / "models" / "Qwen2.5-1.5B-lora"
weight_files = list(ADAPTER.glob("adapter_model*.safetensors")) + list(ADAPTER.glob("*.bin"))
assistant_lora = None
if weight_files:
    assistant_lora = AdvancedRegulationAssistant(
        llm_model="gpt-4o-mini",
        answer_synthesizer_backend="local_qwen",
        qwen_adapter_path=str(ADAPTER),
        qwen_max_new_tokens=1024,
        top_k=TOP_K,
        chunks_root="chunks",
        index_name=INDEX_NAME,
        host=PINECONE_HOST,
        namespace=NAMESPACE,
    )
else:
    print("(d) skipped: no LoRA weights in", ADAPTER)

SYS = "You are a precise regulations assistant for F1 Power Unit Financial Regulations."


def run_a_base_llm(q: str) -> str:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.1,
        messages=[{"role": "system", "content": SYS}, {"role": "user", "content": q}],
    )
    return (r.choices[0].message.content or "").strip()


def run_b_basic_rag(q: str) -> str:
    hits = retriever.retrieve(q, top_k=TOP_K)
    ctx = build_context(hits, max_context_chars=12000)
    if not ctx.strip():
        return "(no retrieval context)"
    return answer_with_rag(query=q, context=ctx, model="gpt-4o-mini", client=client)


def run_c_advanced_base_qwen(q: str, i: int) -> str:
    out = assistant_base_qwen.ask(q, session_id=f"horizontal-eval-c-{i}")
    return out.answer.strip()


def run_d_advanced_lora(q: str, i: int) -> str:
    if assistant_lora is None:
        return "(skipped — no LoRA)"
    out = assistant_lora.ask(q, session_id=f"horizontal-eval-d-{i}")
    return out.answer.strip()


for qi, q in enumerate(EVAL_QUESTIONS, start=1):
    print("\n" + "=" * 100)
    print(f"Q{qi}/{len(EVAL_QUESTIONS)}: {q}\n")
    print("--- (a) Base LLM (no RAG) ---")
    print(run_a_base_llm(q))
    print("\n--- (b) Basic RAG ---")
    print(run_b_basic_rag(q))
    print("\n--- (c) Advanced RAG · base Qwen synthesizer ---")
    try:
        print(run_c_advanced_base_qwen(q, qi))
    except Exception as exc:
        print(f"(error) {type(exc).__name__}: {exc}")
    print("\n--- (d) Advanced RAG · fine-tuned Qwen synthesizer ---")
    try:
        print(run_d_advanced_lora(q, qi))
    except Exception as exc:
        print(f"(error) {type(exc).__name__}: {exc}")



Q1/8: What are the objectives of the Power Unit Financial Regulations?

--- (a) Base LLM (no RAG) ---
The objectives of the Power Unit Financial Regulations in Formula 1 are primarily focused on ensuring financial sustainability, promoting fair competition, and maintaining the integrity of the sport. Key objectives include:

1. **Cost Control**: To limit the expenditure on power unit development and production, ensuring that teams operate within a defined budget.

2. **Fair Competition**: To create a level playing field among teams, preventing wealthier teams from gaining an unfair advantage through unlimited spending on power unit technology.

3. **Encouragement of Innovation**: To foster innovation within the constraints of the regulations, allowing teams to develop competitive power units while adhering to financial limits.

4. **Transparency**: To enhance transparency in financial dealings related to power units, ensuring that teams report their expenditures accurately and consist

### Qualitative discussion: (a) base LLM → (b) basic RAG → (c) advanced RAG (base Qwen) → (d) advanced RAG (fine-tuned Qwen)

**Grounding and factual accuracy.**  
**(a) GPT-4o-mini without retrieval** sounds authoritative but is *not tied to the corpus*. On objectives it invents a sensible “cost control / fair competition” narrative that only loosely matches the real triple objective in the regulations. On the inaugural-season cap it gives a concrete but **wrong** figure (e.g. €10M), which is a classic failure mode: fluency without evidence.  

**(b) Basic RAG** fixes this for most factual lookups: answers track retrieved text (e.g. **US $190M** for inaugural season, **98% / 95%** Reporting Group rule, bonus cap, finance exclusion, inventory rules, R&D deferral). Chunk references make it **auditable**.  

**(c) Advanced RAG with base Qwen** still uses the same retrieval stack upstream, but the *final* answer is only as good as the small model’s ability to read context and follow the JSON-style synthesizer prompt. It matches the regulation well on **Q1** (objectives) and gives a detailed **Reporting Group** answer on **Q3**, but on **Q2** it **answers the wrong question** (generic description of the cap, not the inaugural-season amount). On **Q4–Q5** it collapses to **“No”**, which is **incorrect** relative to the retrieved rules (bonuses are capped, not fully excluded; finance costs *are* excluded). So (c) can look confident while being **wrong or under-specific** compared to (b).  

**(d) Fine-tuned Qwen** sometimes **excels at extracting structured regulatory text** (e.g. **Q2**: correct **$148.5M / $190M** schedule and inaugural-season line). It can also **hallucinate or drift**: wrong clause IDs (**D5.1.1.k** for employee bonuses), **invented** constraints on finance costs (**Q5**), or **off-topic** snippets (e.g. **Q7–Q8** drifting to generic Cost Cap language instead of inventories / R&D deferral). The outputs also include **generation artifacts** (`answer:`, `<|begin_of_text|>`, truncated lists), which come from small-model decoding and training format—not from RAG itself.

**Specificity, completeness, and citations.**  
**(b)** tends to balance **precision** (numbers, conditions) with **short** answers and explicit **[Chunk n]** pointers.  

**(c)** is uneven: long prose when the model elaborates, but **too terse** when it misfires (**Q4–Q5**).
  
**(d)** can produce **long, clause-like** passages (good for audit) but also **paste adjacent bullets** (Q2 items c–e) that are **not** what the question asked, and **omit** the synthesizer’s clean JSON shape—hurting downstream checks and readability.

**What the “advanced” stack adds (in theory vs these runs).**  
The LangGraph path adds memory, planning, relevance filtering, and reference resolution before synthesis. In these traces, **(b)** already retrieves enough for many questions; the main variance is **whether the local synthesizer preserves that evidence** in the final string. **(c)** shows that a **base** 1.5B model is a weak final reader compared to **gpt-4o-mini** in **(b)**. **(d)** shows that **fine-tuning** can align outputs with **regulatory wording** (big win on Q2) but does not remove **confabulation** or **format glitches** without tighter decoding, post-processing, or continued training on JSON + “answer only what was asked.”

**Practical takeaway.**  
For **trustworthy** QA over this corpus, **basic RAG + OpenAI (b)** is the most reliable baseline in this sample. **(a)** is useful only as a **negative control** showing hallucination risk. **(c)** is a **cost / privacy** path but needs a **stronger** local synthesizer or **hybrid** (e.g. OpenAI retained for synthesis only). **(d)** is promising where it **quotes** the rules correctly but needs **stop tokens / regex cleanup**, **stricter citation validation**, and possibly **retuning** so it does not override correct retrieval with confident wrong law.
